# 립리딩 파이프라인 (Colab)

셀은 위에서 아래로 한 번씩 돌린다. 실험 조합은 저장소의 `scripts/run_experiment.py`가
맡으므로 **이 노트북에서 모델·데이터셋·학습 루프를 고치지 않는다.**

2026-08-20에 노트북에 남은 패치 때문에 실험 두 개가 오염됐다. 매번 새 프로세스로
돌리고 설정을 파일에 기록하는 것이 그 사고의 대책이다.

| 절 | 내용 | 언제 |
|---|---|---|
| 1 | 셋업 | 런타임 시작할 때마다 |
| 2 | 데이터 준비 | 화자를 추가할 때만 |
| 3 | 실험 | 매번 |
| 4 | 요약 | 매번 |
| 5 | 결과 커밋 | 실험이 끝나면 |

**출력을 지우고 저장한다.** 학습 로그가 쌓이면 파일이 수 MB가 되고, 얼굴이 담긴
출력은 공개 저장소에 올라가면 안 된다.

---
## 1. 셋업

Drive를 붙이고, 저장소를 최신으로 맞추고, 학습 데이터를 로컬 디스크로 옮긴다.
40초쯤 걸린다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, time, subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
PROCESSED = DRIVE_ROOT / "processed_f60"
CHECKPOINTS = DRIVE_ROOT / "checkpoints"
REPO = Path("/content/hanium-lipreading")
BRANCH = "develop"

n_drive = len(list(PROCESSED.glob("*.npy")))
print("Drive 전처리본", n_drive, "개")
assert n_drive > 0, "전처리본이 없다"

os.chdir("/content")
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH,
                    "https://github.com/HumanRhoid/hanium-lipreading.git",
                    str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
subprocess.run(["pip", "install", "--quiet", "mediapipe", "opencv-python", "wandb"],
               check=False)

import torch
print("torch", torch.__version__, "· GPU", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "없음")

In [ ]:
# 매니페스트를 새로 만들고, Drive는 느리니 학습 데이터를 로컬 디스크로 옮긴다.
# build_manifest가 clip_path를 "processed/…"로 쓰므로 폴더명을 바꾸면 안 된다.
from scripts.build_manifest import build

MANIFEST = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED, manifest_path=MANIFEST)

DATA_ROOT = Path("/content/data_f60")
LOCAL = DATA_ROOT / "processed"
started = time.time()
if len(list(LOCAL.glob("*.npy"))) != n_drive:
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    shutil.copytree(PROCESSED, LOCAL)

n_local = len(list(LOCAL.glob("*.npy")))
empty = [p.name for p in LOCAL.glob("*.npy") if p.stat().st_size == 0]
print("로컬", n_local, "개 ·", round(time.time() - started), "초 · 빈 파일", len(empty), "개")
assert n_local == n_drive and not empty, "로컬 복사가 불완전하다"

---
## 2. 데이터 준비

**화자를 추가하거나 재촬영본을 넣을 때만 돌린다.** 평소에는 건너뛴다.

### 2-1. 촬영본 점검

전처리 전에 명백한 결함을 잡는다. 얼굴 미검출, 세로 정보 손실, 해상도 혼재,
문구별 클립 수 불균형, 발화 길이 부족을 본다.

입술 크기로 합격을 매기지는 않는다. 2026-08-17에 규격을 충족한 s07이 미달인
s09보다 낮아 크기가 성적을 예측하지 못하는 것이 확인됐다.

`--preview`를 쓰면 크롭 미리보기 PNG가 나온다. **입술이 그대로 남으므로 커밋하지 않는다.**

In [ ]:
RAW = DRIVE_ROOT / "raw_s09"       # 촬영본 폴더
SPEAKER = "s09"

!python scripts/check_footage.py --src "{RAW}" --speaker {SPEAKER}

### 2-2. 전처리

영상을 `.npy`로 바꾼다. 파일명은 `화자_문구_번호.mp4` 규칙을 지켜야 하고
문구 안에 밑줄을 쓰면 안 된다. 이미 있는 `.npy`는 건너뛴다.

기본 60프레임이다. 기존 데이터가 전부 60프레임이라 바꾸면 섞을 수 없다.

In [ ]:
!python -m src.ml.preprocess.vid2npy   --raw-dir "{RAW}"   --processed-dir "{PROCESSED}"

print("전처리 후", len(list(PROCESSED.glob("*.npy"))), "개")
print("1절을 다시 돌려 매니페스트와 로컬 복사를 갱신할 것")

---
## 3. 실험

한 줄이다. 중간에 끊겨도 다시 돌리면 `results/<이름>.json`을 보고 이어간다.
같은 이름에 다른 설정을 이어붙이면 스크립트가 설정을 대조해 중단시킨다.

| 목적 | 인자 |
|---|---|
| 기준선 8화자 | `--name base60 --seeds 42 1 7` |
| s05 제외 | `--name no5 --exclude s05 --seeds 42 1 7` |
| 증강 없이 | `--name noaug --seeds 42 1 7 --no-augment` |
| 에폭 120 | `--name e120 --seeds 42 1 7 --epochs 120` |
| 스모크 런 | `--name smoke --speakers s06 --seeds 42 --epochs 3` |

첫 런 출력의 **학습 파라미터 수**를 확인할 것. 기본 설정은 14.31M이다.

In [ ]:
NAME = "base60"
SEEDS = "42"          # "42 1 7" 로 늘리면 판정선이 0.077에서 0.045로 내려간다
EXTRA = ""            # 예: "--exclude s05" 또는 "--no-augment"

!python scripts/run_experiment.py   --name {NAME} --seeds {SEEDS} {EXTRA}   --manifest "{MANIFEST}" --data-root "{DATA_ROOT}"   --checkpoint-dir "{CHECKPOINTS}"   --wandb-project lipreading

---
## 4. 요약

학습 없이 표만 본다. 로컬 PC에서도 돈다.

```
최고점     학습 중 한 번이라도 닿은 값. 아무도 못 고른다
저장값     3에폭 평균 기준으로 고른 값. 낙관 편향이 있다
마지막     마지막 에폭 값. 발표에 쓸 숫자는 이쪽이다
최저 화자  화자 독립 시스템의 실질 성능
부풀림     최고점 − 마지막. 검증 집합 선택 편향의 크기
학습 포화  학습 정확도가 1.000에 닿는 에폭
판정선     이 크기를 넘어야 노이즈와 구분된다
```

판정선은 조건-화자 한 칸의 시드 표준편차 0.0725에서 계산한다.

```
화자 7 · 시드 1    0.077
화자 7 · 시드 3    0.045
화자 8 · 시드 3    0.042
```

In [ ]:
!python scripts/run_experiment.py --name {NAME} --summary --baseline base60   --manifest "{MANIFEST}"

---
## 5. 결과 커밋

`results/*.json`은 커밋한다. 다음 세션에서 기준선으로 바로 쓴다.
체크포인트와 미리보기 이미지는 커밋하지 않는다.

In [ ]:
!git -C {REPO} add results
!git -C {REPO} -c user.email="ssanta011205@gmail.com" -c user.name="ssant"   commit -m "Exp: {NAME}"
!git -C {REPO} push origin {BRANCH}